# Compresr Quickstart

**Compresr** is intelligent LLM context compression. It removes the least-important tokens from a passage while preserving meaning — typically cutting input cost by **30 to 70%** with no model change required.

This tutorial walks through a realistic SaaS customer-support scenario: a long product / industry knowledge corpus (~45k tokens, stitched live from 12 Wikipedia articles) is fed to a model to answer one user question. The same question is answered twice with the same model (`gpt-4o-mini`) once on the full corpus and once on a Compresr-compressed version, so you can see the dollar savings, verify the answers match, and see exactly which tokens Compresr dropped via a GitHub-style word-level diff.

This tutorial covers the core `CompressionClient` SDK with the `latte_v1` query-aware compression model.

> For framework integrations (LangChain, LangGraph, LlamaIndex) see the companion tutorials in this folder.

## 1. Install

In [1]:
%pip install -q -e ".." python-dotenv openai requests ipython


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 2. Load the API key from `.env`

The SDK repo root has a `.env` with `COMPRESR_API_KEY=cmp_...`. The
snippet below walks up until it finds one.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        break

assert os.environ.get("COMPRESR_API_KEY"), "COMPRESR_API_KEY not set"
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not set"
print("API keys loaded.")

API keys loaded.


## 3. Query-aware compression (`latte_v2`)

We stitch 12 SaaS / cloud / subscription-billing Wikipedia articles into one large knowledge corpus (~45k tokens) — the kind of dump a support tool might return when given a vague topic. The customer's actual question is narrow. Watch `latte_v1` drop everything not relevant to that question.

In [3]:
from compresr import CompressionClient
from openai import OpenAI
from IPython.display import display
from _demo_utils import fetch_corpus, compresr_diff_html, print_savings_table

client = CompressionClient(api_key=os.environ["COMPRESR_API_KEY"])
oai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

CORPUS_TITLES = [
    "Software as a service",
    "Cloud computing",
    "Subscription business model",
    "Multitenancy",
    "Customer relationship management",
    "Software industry",
    "Application service provider",
    "Enterprise software",
    "Web application",
    "Software development",
    "Information technology",
    "Computer software",
]
ARTICLE = fetch_corpus(CORPUS_TITLES)
QUESTION = (
    "According to these articles, what happens to a SaaS business if a "
    "significant number of customers cancel their subscriptions, and why do "
    "SaaS companies offer freemium tiers?"
)
SYSTEM = (
    "You are a SaaS customer-support analyst. Answer using ONLY the "
    "provided knowledge corpus. Be concise (2-3 sentences)."
)

def ask(article_text: str) -> tuple[str, int]:
    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"CORPUS:\n{article_text}\n\nQUESTION: {QUESTION}"},
        ],
    )
    return resp.choices[0].message.content, resp.usage.prompt_tokens

print(f"Corpus: {len(ARTICLE):,} chars (~{len(ARTICLE)//4:,} tokens) across {len(CORPUS_TITLES)} articles")
print(f"Question: {QUESTION}\n")

uncompressed_answer, uncompressed_tokens = ask(ARTICLE)

result = client.compress(
    context=ARTICLE,
    query=QUESTION,
    target_compression_ratio=0.5,
)
compressed_text = result.data.compressed_context
compressed_answer, compressed_tokens = ask(compressed_text)

saved_tokens = uncompressed_tokens - compressed_tokens
pct_saved = saved_tokens / uncompressed_tokens * 100
dollars_per_1k_reqs = saved_tokens * 1_000 * 0.15 / 1_000_000

print(f"Compresr  orig -> comp tokens : {result.data.original_tokens:,} -> {result.data.compressed_tokens:,}")
print(f"Compresr  ratio achieved      : {result.data.actual_compression_ratio:.2f}")
print()
print(f"gpt-4o-mini prompt tokens     : {uncompressed_tokens:,} (full) -> {compressed_tokens:,} (compressed)")
print(f"Input tokens saved            : {saved_tokens:,}  ({pct_saved:.1f}%)")
print(f"$ saved per 1,000 requests    : ${dollars_per_1k_reqs:,.2f}  (gpt-4o-mini input @ $0.15/M)")
print()
print("--- Answer WITHOUT compression ---")
print(uncompressed_answer)
print()
print("--- Answer WITH compression ---")
print(compressed_answer)
print()
print_savings_table(uncompressed_tokens, compressed_tokens)
print()
print("Word-level diff (showing first ~18k chars — green = kept, red = dropped):")
display(compresr_diff_html(ARTICLE, compressed_text))

Corpus: 173,709 chars (~43,427 tokens) across 12 articles
Question: According to these articles, what happens to a SaaS business if a significant number of customers cancel their subscriptions, and why do SaaS companies offer freemium tiers?



Compresr  orig -> comp tokens : 31,599 -> 16,145
Compresr  ratio achieved      : 0.49

gpt-4o-mini prompt tokens     : 31,535 (full) -> 16,155 (compressed)
Input tokens saved            : 15,380  (48.8%)
$ saved per 1,000 requests    : $2.31  (gpt-4o-mini input @ $0.15/M)

--- Answer WITHOUT compression ---
If a significant number of customers cancel their subscriptions, the viability of the SaaS business can be jeopardized due to the reliance on recurring revenue from subscriptions. SaaS companies offer freemium tiers to capture a higher market share and attract customers who may eventually convert to paid versions, even if they never upgrade, helping to offset hosting costs associated with a larger user base.

--- Answer WITH compression ---
If a significant number of customers cancel their subscriptions, the viability of the SaaS business can be jeopardized due to the reliance on recurring revenue streams. SaaS companies offer freemium tiers to capture a higher market share and attr

## 4. Batch compression

Two equivalent input forms — the SDK builds the same wire payload
(`inputs: [{context, query}, ...]`) for both:

**Form A (convenience):** pass `contexts` plus `queries` as a single
string (broadcast to all) or a list (one per context).

**Form B (pair list, matches wire format):** pass `inputs` as a list
of `{context, query}` dicts.

In [4]:
docs = [ARTICLE[:len(ARTICLE)//3], ARTICLE[len(ARTICLE)//3:2*len(ARTICLE)//3], ARTICLE[2*len(ARTICLE)//3:]]
batch_a = client.compress_batch(
    contexts=docs,
    queries="How is SaaS typically priced and billed?",
    target_compression_ratio=0.5,
)
print(f"[A] {batch_a.data.count} segments - saved {batch_a.data.total_tokens_saved:,} tokens")

[A] 3 segments - saved 15,286 tokens


In [5]:
batch_b = client.compress_batch(
    inputs=[
        {"context": docs[0], "query": "How is SaaS typically priced?"},
        {"context": docs[1], "query": "What are the legal and compliance challenges of SaaS?"},
        {"context": docs[2], "query": "What architectural patterns are common in SaaS?"},
    ],
    target_compression_ratio=0.5,
)
print(f"[B] {batch_b.data.count} segments - saved {batch_b.data.total_tokens_saved:,} tokens")

[B] 3 segments - saved 15,218 tokens


## 5. What does this save?

The metrics line printed above is the real measurement for **one** support query against this ~45k-token corpus. A typical SaaS support backend runs **thousands of such queries per day**.

At gpt-4o-mini input pricing (~USD 0.15 per million tokens), saving roughly half of every input on a 10M-token/day workload is **about USD 22 per day per tenant**. On a frontier model like gpt-4o (~USD 2.50 per million input) the same workload saves **about USD 375 per day**. The compression call itself is a single fast batch round-trip, not a synchronous per-message penalty.

Importantly, the two answers above were generated by the *same* model on the *same* question; the only difference is the corpus was shrunk by ~half before the second call. The word-level diff above shows exactly which tokens were dropped — Compresr keeps the answer-relevant phrases and removes the rest.

**Next**: pick the framework you use and read the matching tutorial:

- `02_langchain.ipynb` - wrap_tool, middleware (tool / summary / prompt), CompresrExtractor
- `03_langgraph.ipynb` - graph nodes, middleware re-exports, checkpoint serializer, handoff tool, store wrapper
- `04_llamaindex.ipynb` - node postprocessor for RAG, tool wrapper, memory block
- `05_compresr_agents.ipynb` - web search + tool-output compression for agentic flows